# Train the Lift world model and goal-conditioned actor

Run the cells in order on a Colab GPU runtime. Choose your Hugging Face dataset
below; training data is downloaded to local runtime storage. Google Drive stores
checkpoints and logs only. Configure training in the final training cell.

Training jointly optimizes prediction MSE, SIGReg, and actor negative log-likelihood.
Epochs are additional when resuming. The best checkpoint uses validation total loss,
which is not a measure of task success.


In [ ]:
%pip install -q -U huggingface_hub hf_xet h5py tensorboard pyyaml tqdm wandb


## Dataset and output storage

Set your dataset repository below to `USERNAME/DATASET`. It must contain
`demo_actor.hdf5` produced by `processing_actor.py`.
Public datasets need no token. For private datasets, use Colab secret `HF_TOKEN`,
a saved Hugging Face login, or the login prompt. A login on your own computer
does not automatically carry over to Colab.

Drive folders `models/` and `logs/` are created inside `DRIVE_DIR`.
No evaluation files are needed for training. The download has a progress indicator
and is reused on the same runtime. Pin `HF_REVISION` to a commit SHA for reproducibility.

Optional W&B logging uses Colab secret `WANDB_TOKEN`; TensorBoard remains available.

### Set up Colab Secrets before running

1. Open the key icon (**Secrets**) in Colab's left sidebar.
2. Add a secret named **HF_TOKEN** with your Hugging Face access token.
   It needs read access to the chosen dataset; public datasets do not require it.
3. Add a secret named **WANDB_TOKEN** with your W&B API key.
4. Turn on **Notebook access** for each secret and approve access if prompted.
   Names are case-sensitive. Never paste tokens into notebook code or outputs.

Get your Hugging Face token from your account's Settings → Access Tokens.
Get your W&B API key from your W&B account settings.

**Hugging Face:** the download cell tries HF_TOKEN, then a saved login.
If anonymous access fails and no token is available, it opens a login prompt.
A saved login on your personal computer is not inherited by a Colab runtime.

**W&B:** authentication happens when you run the training cell. In Colab,
WANDB_TOKEN is required when ENABLE_WANDB=True; there is no fallback token prompt.
Set ENABLE_WANDB=False to skip W&B and keep TensorBoard logging.

Secrets are stored in your Colab account, not in the shared notebook. Other users
must supply their own secrets. You normally add each secret once, but must allow
access for this notebook.


In [ ]:
from pathlib import Path

HF_REPO = 'YOUR_USERNAME/YOUR_DATASET'  # Hugging Face dataset repository, not a URL.
HF_REVISION = 'main'                   # Branch, tag, or pinned dataset commit SHA.
DRIVE_DIR = Path('/content/drive/MyDrive/primary_actor')  # Checkpoints and logs.


In [ ]:
import shutil
import h5py
from google.colab import drive, userdata
from huggingface_hub import HfApi, get_token, login, hf_hub_download
from huggingface_hub.errors import HfHubHTTPError

if HF_REPO == 'YOUR_USERNAME/YOUR_DATASET' or len(HF_REPO.split('/')) != 2:
    raise ValueError('Set HF_REPO above to your Hugging Face dataset repository')
drive.mount('/content/drive')
try:
    token = userdata.get('HF_TOKEN') or get_token()
except Exception:
    token = get_token()
try:
    info = HfApi(token=token).dataset_info(HF_REPO, revision=HF_REVISION, files_metadata=True)
except HfHubHTTPError as error:
    if token or error.response is None or error.response.status_code not in (401, 403, 404):
        raise
    print('Dataset unavailable anonymously. Log in with an account that has access; also check HF_REPO.')
    login(add_to_git_credential=False)
    token = get_token()
    info = HfApi(token=token).dataset_info(HF_REPO, revision=HF_REVISION, files_metadata=True)
DATASET_REVISION = info.sha
LOCAL_DIR = Path('/content/data/robot/actor')
LOCAL_DIR.mkdir(parents=True, exist_ok=True)
file_info = next((item for item in info.siblings if item.rfilename == 'demo_actor.hdf5'), None)
if file_info is None:
    raise FileNotFoundError(f'{HF_REPO} does not contain demo_actor.hdf5')
local_file = LOCAL_DIR / 'demo_actor.hdf5'
if not local_file.exists() and file_info.size is not None and shutil.disk_usage(LOCAL_DIR).free < file_info.size * 1.1:
    raise OSError('Not enough local disk space for the actor dataset')
DATASET_PATH = Path(hf_hub_download(
    repo_id=HF_REPO, repo_type='dataset', filename='demo_actor.hdf5',
    revision=DATASET_REVISION, local_dir=LOCAL_DIR, token=token))
del token
with h5py.File(DATASET_PATH, 'r') as f:
    if f.attrs.get('actor_dataset_version') != 1 or f.attrs.get('frame_skip') != 5:
        raise ValueError('Expected a processed actor dataset with version=1 and frame_skip=5')
    print('dataset:', DATASET_PATH, 'revision:', DATASET_REVISION)
    print('episodes:', len(f['data']), 'split sizes:', {k: len(f['mask'][k]) for k in ('train', 'valid')})


In [ ]:
%%writefile /content/actor_dataset_runtime.py
import re
from bisect import bisect_right
from pathlib import Path

import h5py
import numpy as np
import torch
from torch.utils.data import Dataset

"""
PyTorch dataset over MimicGen robot demos (robomimic-style HDF5).

Each sample contains a strided observation sequence and one ordered action block for
each transition:

  observations: [o_t, o_{t+s}, ..., o_{t + window*s}]
  actions:      [[a_t, ..., a_{t+s-1}], ...]

Here ``s`` is ``frame_skip``. Every primitive command is retained in temporal
order; no averaging or accumulation is performed. Proprio
state is eef_pos (3) + eef_quat (4) + gripper_qpos (2) = 9D. Images are
returned as uint8 CHW tensors and ImageNet-normalized on the training device.

Train vs val is selected via the HDF5 mask group (`mask/train`, `mask/valid`).
Both splits use state statistics computed only from the training demonstrations.
"""


def _demo_number(name: str) -> int:
	return int(re.search(r"(\d+)", name).group(1))


def _decode_demo_name(raw) -> str:
	if isinstance(raw, bytes):
		return raw.decode("utf-8")
	return str(raw)


def build_robot_state(pos, quat, grip):
	"""Concatenate normalized robosuite proprioception tensors."""
	quat = quat / quat.norm(dim=-1, keepdim=True).clamp_min(1e-8)
	# Preserve robosuite's temporally continuous quaternion sign. Forcing a
	# fixed hemisphere creates artificial jumps when a trajectory crosses w=0.
	return torch.cat((pos, quat, grip), dim=-1)


def normalize_robot_state(state, state_mean, state_std):
	return (state - state_mean) / state_std


def normalize_robot_images(images):
	"""Apply the ImageNet normalization used by the released LeWM encoder."""
	if images.shape[-3] != 3:
		raise ValueError(f"expected RGB images, got shape {tuple(images.shape)}")
	shape = [1] * images.ndim
	shape[-3] = 3
	mean = images.new_tensor((0.485, 0.456, 0.406)).reshape(shape)
	std = images.new_tensor((0.229, 0.224, 0.225)).reshape(shape)
	return (images - mean) / std


def compute_train_state_stats(h5_path: Path) -> tuple[torch.Tensor, torch.Tensor]:
	"""Compute 9-D proprioception statistics from training demos only."""
	count = 0
	total = torch.zeros(9, dtype=torch.float64)
	total_sq = torch.zeros(9, dtype=torch.float64)

	with h5py.File(h5_path, "r") as f:
		episodes = [_decode_demo_name(x) for x in f["mask/train"][:]]
		for ep_name in episodes:
			obs = f[f"data/{ep_name}/obs"]
			state = build_robot_state(
				torch.from_numpy(obs["robot0_eef_pos"][:]).to(torch.float64),
				torch.from_numpy(obs["robot0_eef_quat"][:]).to(torch.float64),
				torch.from_numpy(obs["robot0_gripper_qpos"][:]).to(torch.float64),
			)
			count += state.shape[0]
			total += state.sum(dim=0)
			total_sq += state.square().sum(dim=0)

	if count == 0:
		raise ValueError(f"No training observations found in {h5_path}")

	mean = total / count
	variance = (total_sq / count) - mean.square()
	std = variance.clamp_min(1e-12).sqrt().clamp_min(1e-6)
	return mean.float(), std.float()


def compute_train_action_stats(h5_path: Path):
	"""Compute primitive-action statistics and robust bounds from train demos."""
	action_arrays = []
	with h5py.File(h5_path, "r") as f:
		episodes = [_decode_demo_name(x) for x in f["mask/train"][:]]
		for ep_name in episodes:
			actions = np.asarray(f[f"data/{ep_name}/actions"][:], dtype=np.float32)
			if actions.ndim != 2:
				raise ValueError(f"{ep_name} actions must be rank 2, got {actions.shape}")
			action_arrays.append(actions)

	if not action_arrays:
		raise ValueError(f"No training actions found in {h5_path}")
	actions = np.concatenate(action_arrays, axis=0)
	mean = torch.from_numpy(actions.mean(axis=0, dtype=np.float64).astype(np.float32))
	std = torch.from_numpy(actions.std(axis=0, dtype=np.float64).astype(np.float32))
	std = std.clamp_min(1e-6)
	low = torch.from_numpy(np.quantile(actions, 0.005, axis=0).astype(np.float32))
	high = torch.from_numpy(np.quantile(actions, 0.995, axis=0).astype(np.float32))
	# Zero is a useful no-motion proposal and remains inside robosuite's bounds.
	low = torch.minimum(low, torch.zeros_like(low))
	high = torch.maximum(high, torch.zeros_like(high))
	degenerate = (high - low) < 1e-6
	low = torch.where(degenerate, mean - std, low)
	high = torch.where(degenerate, mean + std, high)
	return mean, std, low, high


class RobotPrimaryDataset(Dataset):
	def __init__(
		self,
		data_dir: Path,
		frame_skip: int,
		window: int,
		val: bool = False,
		state_mean: torch.Tensor | None = None,
		state_std: torch.Tensor | None = None,
		action_mean: torch.Tensor | None = None,
		action_std: torch.Tensor | None = None,
		h5_path: Path | None = None,
	):
		super().__init__()
		if frame_skip < 1:
			raise ValueError(f"frame_skip must be positive, got {frame_skip}")
		if window < 1:
			raise ValueError(f"window must be positive, got {window}")

		self.frame_skip = frame_skip
		self.window = window
		self.h5_path = Path(h5_path) if h5_path is not None else Path(data_dir) / "robot" / "mimicgen" / "demo.hdf5"
		if not self.h5_path.exists():
			raise FileNotFoundError(f"Dataset not found: {self.h5_path}")

		if state_mean is None or state_std is None:
			state_mean, state_std = compute_train_state_stats(self.h5_path)
		self.state_mean = torch.as_tensor(state_mean, dtype=torch.float32)
		self.state_std = torch.as_tensor(state_std, dtype=torch.float32)
		if self.state_mean.shape != (9,) or self.state_std.shape != (9,):
			raise ValueError("state_mean and state_std must both have shape (9,)")
		if action_mean is None or action_std is None:
			action_mean, action_std, _, _ = compute_train_action_stats(self.h5_path)
		self.action_mean = torch.as_tensor(action_mean, dtype=torch.float32)
		self.action_std = torch.as_tensor(action_std, dtype=torch.float32)
		if self.action_mean.ndim != 1 or self.action_std.shape != self.action_mean.shape:
			raise ValueError("action_mean and action_std must be matching vectors")

		mask_key = "valid" if val else "train"
		self.episodes = []
		self.cumulative_windows = []
		total_windows = 0
		with h5py.File(self.h5_path, "r") as f:
			episodes = [_decode_demo_name(x) for x in f["mask"][mask_key][:]]
			episodes = sorted(episodes, key=_demo_number)

			for ep_name in episodes:
				ep_len = int(
					f[f"data/{ep_name}"].attrs.get(
						"num_samples",
						f[f"data/{ep_name}/actions"].shape[0],
					)
				)
				num_windows = max(0, ep_len - window * frame_skip)
				if num_windows:
					total_windows += num_windows
					self.episodes.append(ep_name)
					self.cumulative_windows.append(total_windows)

		# Keep one cumulative count per episode instead of one Python tuple and
		# frame list per window. This stays small even for millions of windows.
		self.total_len = total_windows
		self._h5 = None

	def __len__(self):
		return self.total_len

	def _get_h5(self):
		# DataLoader workers each lazily open their own read-only handle.
		if self._h5 is None:
			self._h5 = h5py.File(self.h5_path, "r")
		return self._h5

	def _resolve_index(self, idx):
		if idx < 0:
			idx += self.total_len
		if idx < 0 or idx >= self.total_len:
			raise IndexError(idx)

		episode_idx = bisect_right(self.cumulative_windows, idx)
		episode_start = (
			0 if episode_idx == 0 else self.cumulative_windows[episode_idx - 1]
		)
		return self.episodes[episode_idx], idx - episode_start

	def __getstate__(self):
		state = self.__dict__.copy()
		state["_h5"] = None
		return state

	def close(self):
		if getattr(self, "_h5", None) is not None:
			self._h5.close()
			self._h5 = None

	def __del__(self):
		self.close()

	def _load_images(self, obs, frame_slice, cam_key):
		# HDF5 stepped hyperslabs are several times slower than reading the
		# already-touched temporal chunks contiguously and subsampling in memory.
		imgs = torch.from_numpy(
			obs[cam_key][frame_slice.start:frame_slice.stop][::frame_slice.step]
		)
		return imgs.permute(0, 3, 1, 2)

	def _load_state(self, obs, frame_slice):
		contiguous_slice = slice(frame_slice.start, frame_slice.stop)
		step = frame_slice.step
		state = build_robot_state(
			torch.from_numpy(obs["robot0_eef_pos"][contiguous_slice][::step]).float(),
			torch.from_numpy(obs["robot0_eef_quat"][contiguous_slice][::step]).float(),
			torch.from_numpy(obs["robot0_gripper_qpos"][contiguous_slice][::step]).float(),
		)
		return normalize_robot_state(state, self.state_mean, self.state_std)

	def __getitem__(self, idx):
		ep_name, start = self._resolve_index(idx)
		f = self._get_h5()
		episode = f["data"][ep_name]
		obs = episode["obs"]
		frame_slice = slice(
			start,
			start + self.window * self.frame_skip + 1,
			self.frame_skip,
		)

		above = self._load_images(obs, frame_slice, "agentview_image")
		wrist = self._load_images(obs, frame_slice, "robot0_eye_in_hand_image")
		state = self._load_state(obs, frame_slice)

		# Each transition keeps all frame_skip primitive OSC commands in order.
		primitive_actions = torch.from_numpy(
			episode["actions"][
				start:start + self.window * self.frame_skip
			]
		).float().reshape(self.window, self.frame_skip, -1)
		if primitive_actions.shape[-1] != self.action_mean.numel():
			raise ValueError(
				f"dataset actions have size {primitive_actions.shape[-1]}, "
				f"statistics have size {self.action_mean.numel()}"
			)
		actions = (primitive_actions - self.action_mean) / self.action_std
		return above, wrist, state, actions


"""Spawn-safe loader for processing_actor.py's indexed HDF5 schema."""
import h5py
import numpy as np
import torch
from pathlib import Path



class RobotActorDataset(RobotPrimaryDataset):
    def __init__(self, data_dir, *args, **kwargs):
        kwargs.setdefault('h5_path', Path(data_dir) / 'robot/actor/demo_actor.hdf5')
        super().__init__(data_dir, *args, **kwargs)
        if self.window != 3:
            raise ValueError('Actor dataset has three transitions per sample')
        # Compact tables are loaded once, not repeatedly decompressed per sample.
        observations, actions, masks = [], [], []
        with h5py.File(self.h5_path, 'r') as f:
            if f.attrs.get('actor_dataset_version') != 1 or f.attrs.get('frame_skip') != self.frame_skip:
                raise ValueError('Incompatible actor dataset version or frame skip')
            for name in self.episodes:
                group = f[f'data/{name}/actor_samples']
                observations.append(group['observation_indices'][:])
                actions.append(group['action_start_indices'][:])
                masks.append(group['actor_action_mask'][:])
        if not observations:
            raise ValueError('Empty actor dataset split')
        self.observation_indices = np.concatenate(observations)
        self.action_indices = np.concatenate(actions)
        self.actor_masks = np.concatenate(masks)
        if self.observation_indices.shape != (len(self), 5):
            raise ValueError('Actor sample counts differ from chronological windows')
        if self.action_indices.shape != (len(self), 3) or self.actor_masks.shape != (len(self), self.frame_skip):
            raise ValueError('Malformed actor action indices/masks')
        if not np.all(np.diff(self.observation_indices[:, :4], axis=1) == self.frame_skip):
            raise ValueError('Nonconsecutive world-model indices')
        if not np.array_equal(self.action_indices, self.observation_indices[:, :3]):
            raise ValueError('Misaligned primitive action blocks')
        expected = (self.observation_indices[:, 2, None] + np.arange(self.frame_skip)
                    < self.observation_indices[:, 4, None])
        if not np.array_equal(expected, self.actor_masks) or not self.actor_masks.any(axis=1).all():
            raise ValueError('Invalid actor boundary masks')

    def __getitem__(self, idx):
        if idx < 0:
            idx += len(self)
        name, offset = self._resolve_index(idx)
        indices = self.observation_indices[idx]
        start, goal = int(indices[0]), int(indices[-1])
        if start != offset:
            raise ValueError('Unexpected sample ordering')
        episode = self._get_h5()[f'data/{name}']
        obs = episode['obs']
        end = int(indices[3]) + 1
        def frames(key):
            # Retain contiguous-read optimization; goal can precede last frame
            # or duplicate a history frame, so do not use HDF5 fancy indexing.
            history = obs[key][start:end][::self.frame_skip]
            if goal in indices[:4]:
                target = history[np.flatnonzero(indices[:4] == goal)[0]]
            else:
                target = obs[key][goal]
            return torch.from_numpy(np.concatenate((history, target[None]), axis=0))
        above = frames('agentview_image').permute(0, 3, 1, 2)
        wrist = frames('robot0_eye_in_hand_image').permute(0, 3, 1, 2)
        # Reuse the base loader's quaternion handling and normalization.
        from_state = self._load_state(obs, slice(start, end, self.frame_skip))
        goal_state = self._load_state(obs, slice(goal, goal + 1, 1))
        state = torch.cat((from_state, goal_state), dim=0)
        actions = torch.from_numpy(episode['actions'][start:start + 3*self.frame_skip]).float()
        actions = ((actions - self.action_mean) / self.action_std).reshape(3, self.frame_skip, -1)
        return above, wrist, state, actions, torch.from_numpy(self.actor_masks[idx].copy())


In [ ]:
import importlib
import sys
from pathlib import Path

module_dir = Path('/content')
if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))
importlib.invalidate_caches()
sys.modules.pop('actor_dataset_runtime', None)
_spawn_module = importlib.import_module('actor_dataset_runtime')

RobotPrimaryDataset = _spawn_module.RobotPrimaryDataset
compute_train_state_stats = _spawn_module.compute_train_state_stats
compute_train_action_stats = _spawn_module.compute_train_action_stats
build_robot_state = _spawn_module.build_robot_state
normalize_robot_state = _spawn_module.normalize_robot_state
normalize_robot_images = _spawn_module.normalize_robot_images
print('spawn-safe dataset module ready: /content/actor_dataset_runtime.py')

RobotActorDataset = _spawn_module.RobotActorDataset


In [ ]:
import torch
import torch.nn.functional as F
from torch import nn


class ProjectionMLP(nn.Module):
    """LeWM projection head used after encoder and predictor features."""

    def __init__(self, input_dim: int = 192, hidden_dim: int = 2048, output_dim: int = 192):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x):
        return self.layers(x)


class ActionEncoder(nn.Module):
    """Encode one ordered primitive-action block for AdaLN conditioning."""

    def __init__(self, input_dim: int, smoothed_dim: int = 10, embedding_dim: int = 192):
        super().__init__()
        self.smoother = nn.Conv1d(input_dim, smoothed_dim, kernel_size=1)
        self.mlp = nn.Sequential(
            nn.Linear(smoothed_dim, 4 * embedding_dim),
            nn.SiLU(),
            nn.Linear(4 * embedding_dim, embedding_dim),
        )

    def forward(self, actions):
        batch_size, sequence_length, _ = actions.shape
        actions = self.smoother(actions.transpose(1, 2)).transpose(1, 2)
        return self.mlp(actions.reshape(batch_size * sequence_length, -1)).reshape(
            batch_size, sequence_length, -1
        )


class TransformerBlock(nn.Module):
    """Pre-norm transformer block with optional AdaLN-Zero conditioning."""

    def __init__(
        self,
        d_model: int = 192,
        nheads: int = 16,
        d_action: int | None = None,
        dropout: float = 0.1,
        causal_masking: bool = True,
        adaptive: bool = True,
        dim_head: int | None = None,
        mlp_dim: int | None = None,
    ):
        super().__init__()
        self.adaptive = adaptive
        hidden_dim = mlp_dim or 4 * d_model

        self.norm_one = nn.LayerNorm(d_model, elementwise_affine=not adaptive)
        self.norm_two = nn.LayerNorm(d_model, elementwise_affine=not adaptive)
        if adaptive:
            if d_action is None:
                raise ValueError("d_action is required for an adaptive transformer block")
            self.adaLN_modulation = nn.Sequential(
                nn.SiLU(),
                nn.Linear(d_action, 6 * d_model),
            )
            nn.init.zeros_(self.adaLN_modulation[-1].weight)
            nn.init.zeros_(self.adaLN_modulation[-1].bias)

        self.attn_block = MultiHeadAttention(
            d_model=d_model,
            nheads=nheads,
            dim_head=dim_head,
            dropout=dropout,
            causal_masking=causal_masking,
        )
        self.mlp = nn.Sequential(
            nn.Linear(d_model, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, d_model),
            nn.Dropout(dropout),
        )

    @staticmethod
    def _modulate(x, shift, scale):
        return x * (1 + scale) + shift

    def forward(self, z, a=None):
        if self.adaptive:
            if a is None:
                raise ValueError("action conditioning is required for an adaptive block")
            shift_attn, scale_attn, gate_attn, shift_mlp, scale_mlp, gate_mlp = (
                self.adaLN_modulation(a).chunk(6, dim=-1)
            )
            z = z + gate_attn * self.attn_block(
                self._modulate(self.norm_one(z), shift_attn, scale_attn)
            )
            z = z + gate_mlp * self.mlp(
                self._modulate(self.norm_two(z), shift_mlp, scale_mlp)
            )
        else:
            z = z + self.attn_block(self.norm_one(z))
            z = z + self.mlp(self.norm_two(z))
        return z


class MultiHeadAttention(nn.Module):
    def __init__(
        self,
        d_model: int = 192,
        nheads: int = 16,
        dim_head: int | None = None,
        dropout: float = 0.1,
        causal_masking: bool = True,
    ):
        super().__init__()
        self.dropout = dropout
        self.d_k = dim_head or (d_model // nheads)
        self.nheads = nheads
        self.inner_dim = self.nheads * self.d_k
        self.causal_masking = causal_masking

        self.W_q = nn.Linear(d_model, self.inner_dim)
        self.W_k = nn.Linear(d_model, self.inner_dim)
        self.W_v = nn.Linear(d_model, self.inner_dim)
        self.W_o = nn.Linear(self.inner_dim, d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, sequence_length, _ = x.shape
        queries = self.W_q(x).reshape(
            batch_size, sequence_length, self.nheads, self.d_k
        ).transpose(1, 2)
        keys = self.W_k(x).reshape(
            batch_size, sequence_length, self.nheads, self.d_k
        ).transpose(1, 2)
        values = self.W_v(x).reshape(
            batch_size, sequence_length, self.nheads, self.d_k
        ).transpose(1, 2)

        attention = F.scaled_dot_product_attention(
            queries,
            keys,
            values,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=self.causal_masking,
        )
        attention = attention.transpose(1, 2).reshape(
            batch_size, sequence_length, self.inner_dim
        )
        return self.drop(self.W_o(attention))


In [ ]:
from torch import nn
import torch


class CustomVitTiny(nn.Module):
    def __init__(self,
                 patch : int = 14,
                 nheads : int = 3,
                 nlayers : int = 12,
                 d_model : int = 192,
                 npatches : int = 256):
        super().__init__()

        self.npatches = npatches
        self.d_model = d_model

        self.conv_flattener = nn.Conv2d(in_channels=3,
                                        out_channels=d_model,
                                        kernel_size=patch,
                                        stride=patch,
                                        padding=0)
        
        self.transformer_layers = nn.ModuleList([])
        for _ in range(nlayers):
            self.transformer_layers.append(TransformerBlock(d_model=d_model, 
                                           nheads=nheads,                                            
                                           dropout=0.0,
                                           causal_masking=False,
                                           adaptive=False))
        # learned positional embeddings
        self.position_emb = nn.Parameter(torch.empty(npatches + 1, d_model))
        # learned [cls] embedding
        self.cls_emb = nn.Parameter(torch.empty(1, 1, 1, d_model))
        self.norm = nn.LayerNorm(d_model)
        nn.init.trunc_normal_(self.position_emb, std=0.02)
        nn.init.trunc_normal_(self.cls_emb, std=0.02)

    def forward(self, x):
        # shape of x: (B, N, C, H, W)
        B, N, C, H, W = x.shape
        x = x.reshape(B*N, C, H, W)
        emb = self.conv_flattener(x)
        emb = emb.reshape(B, N, self.d_model, self.npatches).permute(0, 1, 3, 2)
        # shape of emb now: (512, 3, 256, 192)
        cls_emb = self.cls_emb.expand(B, N, 1, self.d_model)
        emb = torch.cat([cls_emb, emb], dim=2)
        emb = emb + self.position_emb
        # shape of emb now: (512, 3, 257, 192)
        emb = emb.reshape(B*N, self.npatches + 1, self.d_model) # now ready to go into the transformer

        for layer in (self.transformer_layers):
           emb = layer(emb)
        emb = self.norm(emb)

        emb = emb.reshape(B, N, self.npatches + 1, self.d_model)
        emb = emb[:, :, 0, :] # shape : (512, 3, 192), this is what the predictor expects as the input

        return emb





# TEST CODE for Shape
# dummy_conv = nn.Conv2d(in_channels=3,
#           out_channels=192,
#           kernel_size=14,
#           stride=14,
#           padding=0)

# dummy_frame = torch.ones((512, 3, 3, 224, 224))
# dummy_frame = dummy_frame.reshape(512*3, 3, 224, 224)
# emb = dummy_conv(dummy_frame)

# print(emb.shape)
# print("after reshaping")
# emb = emb.reshape((512, 3, 192, 256))
# emb = emb.permute(0, 1, 3, 2)
# print(emb.shape)
# cls = torch.ones(1, 1, 1, 192)
# cls = cls.expand(512, 3, 1, cls.shape[-1])
# emb = torch.cat([cls, emb], dim=2)
# print(emb.shape)
# emb = emb[:, :, 0, :]
# print(emb.shape)

# TEST CODE for the Architecture
# test = CustomVitTiny()
# dummy = torch.ones(2, 3, 3, 224, 224)
# dummy = test(dummy)
# print(dummy.shape)


In [ ]:
from torch import nn
import torch

# right now the image size is defaulted to 224x224

class EncoderNetwork(nn.Module):
    def __init__(self,
                 patch : int = 14,
                 nheads : int = 3,
                 nlayers : int = 12,
                 d_model : int = 192,
                 npatches : int = 256,
                 projector_hidden_dim: int = 2048,
                 project_output: bool = True): # npatches is (H/patch_size)*(W/patch_size)
        super().__init__()
        self.d_model = d_model
        
        self.vit = CustomVitTiny(patch, nheads, nlayers, d_model, npatches)
        self.projector = (
            ProjectionMLP(d_model, projector_hidden_dim, d_model)
            if project_output else nn.Identity()
        )
    
    def forward(self, x):
        # shape of input (x): (B, N, C, H, W)
        B, N, C, H, W = x.shape
        z = self.vit(x)
        #projection step
        z = z.reshape(B*N, self.d_model)
        z = self.projector(z)
        z = z.reshape(B, N, self.d_model)

        return z
    

# TEST Code
# dummy_image = torch.ones(24, 3, 3, 224, 224)
# enc = EncoderNetwork()
# print(enc(dummy_image).shape)


In [ ]:
from torch import nn
import torch


class RobotEncoder(nn.Module):
    def __init__(self,
                 patch : int = 14,
                 nheads : int = 3,
                 nlayers : int = 12,
                 d_model : int = 192,
                 d_state : int = 9,
                 d_state_emb : int = 64,
                 npatches : int = 36,
                 projector_hidden_dim: int = 2048):  # 84x84, patch 14 → 6x6 patches
        super().__init__()
        encoder_kwargs = dict(
            patch=patch,
            nheads=nheads,
            nlayers=nlayers,
            d_model=d_model,
            npatches=npatches,
            projector_hidden_dim=projector_hidden_dim,
            project_output=False,
        )
        # The architectures are identical, but no parameters are shared. This
        # lets each visual backbone specialize to its camera's representation.
        self.agent_encoder = EncoderNetwork(**encoder_kwargs)
        self.wrist_encoder = EncoderNetwork(**encoder_kwargs)

        # small projection network for the 
        self.state_encoder = nn.Sequential(
            nn.Linear(in_features=d_state, out_features=2*d_state),
            nn.GELU(),
            nn.Linear(in_features=2*d_state, out_features=d_state_emb)
        )

        self.fusion_mlp = nn.Sequential(
            nn.Linear(in_features=(d_model * 2 + d_state_emb), out_features=(d_model * 2)),
            nn.GELU(),
            nn.Linear(in_features=(d_model * 2), out_features=d_model),
        )
        self.projector = ProjectionMLP(d_model, projector_hidden_dim, d_model)

    def forward(self, above, wrist, state):
        # above : [Batch, Window, 3, 84, 84] | wrist : [Batch, Window, 3, 84, 84] | state : [Batch, Window, 9]
        above_emb = self.agent_encoder(above)
        wrist_emb = self.wrist_encoder(wrist)
        state_emb = self.state_encoder(state) # [Batch, Window, 64]
        fusion = torch.cat((above_emb, wrist_emb, state_emb), dim=-1) # [Batch, Window, 448]
        fusion = self.fusion_mlp(fusion)
        flat = fusion.reshape(-1, fusion.shape[-1])
        return self.projector(flat).reshape_as(fusion) # [Batch, Window, 192]


In [ ]:
import torch
from torch import nn





class PredictorNetwork(nn.Module):
    """Action-conditioned latent predictor matching the released LeWM design."""

    def __init__(
        self,
        d_model: int = 192,
        nheads: int = 16,
        d_action: int = 2,
        dropout: float = 0.1,
        nlayers: int = 6,
        window: int = 3,
        dim_head: int = 64,
        mlp_dim: int = 2048,
        action_smoothing_dim: int = 10,
        projector_hidden_dim: int = 2048,
    ):
        super().__init__()
        self.d_model = d_model
        self.d_action = d_action
        self.window = window
        self.action_encoder = ActionEncoder(
            input_dim=d_action,
            smoothed_dim=action_smoothing_dim,
            embedding_dim=d_model,
        )
        self.transformer_layers = nn.ModuleList(
            [
                TransformerBlock(
                    d_model=d_model,
                    nheads=nheads,
                    d_action=d_model,
                    dropout=dropout,
                    causal_masking=True,
                    adaptive=True,
                    dim_head=dim_head,
                    mlp_dim=mlp_dim,
                )
                for _ in range(nlayers)
            ]
        )
        self.position_emb = nn.Parameter(torch.empty(1, window, d_model))
        nn.init.trunc_normal_(self.position_emb, std=0.02)
        self.final_norm = nn.LayerNorm(d_model)
        self.projector = ProjectionMLP(d_model, projector_hidden_dim, d_model)

    def forward(self, z, a):
        batch_size, sequence_length, embedding_dim = z.shape
        if embedding_dim != self.d_model:
            raise ValueError(f"expected embedding size {self.d_model}, got {embedding_dim}")
        if sequence_length > self.window:
            raise ValueError(f"sequence length {sequence_length} exceeds window {self.window}")
        if a.shape[:2] != z.shape[:2] or a.shape[-1] != self.d_action:
            raise ValueError(
                f"expected actions [B, {sequence_length}, {self.d_action}], got {tuple(a.shape)}"
            )

        z = z + self.position_emb[:, :sequence_length]
        action_embedding = self.action_encoder(a)
        for layer in self.transformer_layers:
            z = layer(z, action_embedding)
        z = self.final_norm(z)
        z = self.projector(z.reshape(batch_size * sequence_length, self.d_model))
        return z.reshape(batch_size, sequence_length, self.d_model)


In [ ]:
import torch
from torch import nn
import numpy as np


class SIGReg(nn.Module):
    def __init__(self,
                 knots : int = 17, # the number of points being used to approximate the integral
                 num_proj : int = 1024): # number of random directions the vector will be projected in (M)
        super().__init__()
        # setting up the trapezoidal approximation of the integral in epps-pulley test

        self.num_proj = num_proj
        # this is the tensor split up into points interpolated between 0 to 3 which will be integrated over
        # only need to 0 to 3 as the gaussian essentially goes to 0 after 3 so this serves as a strong approximate
        t = torch.linspace(0, 3, knots, dtype=torch.float32)
        dt = 3 / (knots - 1)

        # these are the weights based on the trapezoid approximation [f(x0) + 2f(x1) + ... 2f(xn-1) + f(xn)]
        weights = torch.full((knots,), 2 * dt, dtype=torch.float32)
        weights[[0, -1]] = dt

        # gaussian characteristic function
        window = torch.exp(-t.square() / 2.0)

        # storing relevant values in buffer as non learnable parameters so that they move across devices with the nn Module
        self.register_buffer("t", t)
        self.register_buffer("phi", window)
        self.register_buffer("weights", weights*window)

    def forward(self, Z): # Z : (N, B, D) or (N+1, B, D) if target embeddins is being included
        A = torch.randn(Z.size(-1), self.num_proj, device=Z.device) # tensor of random directions that the embeddings will be projected onto
        A = A.div_(A.norm(p=2, dim=0)) # normalizing the tensor to be unit vectors

        # computing the epps-pulley stat
        x_t = (Z @ A).unsqueeze(-1) * self.t
        err = (x_t.cos().mean(-3) - self.phi).square() + x_t.sin().mean(-3).square()
        stat = (err @ self.weights) * Z.size(-2)

        return stat.mean() # returning the mean error across each projection


In [ ]:
import contextlib

import torch


def resolve_amp_dtype(amp_dtype: str, device) -> torch.dtype | None:
	"""Resolve autocast dtype from config.

	Supported values:
	  - auto: bf16 on GPUs with native BF16 support (Ampere+), else fp16 on CUDA
	  - bf16 / bfloat16
	  - fp16 / float16
	  - off / none: disable autocast
	"""
	if isinstance(device, str):
		on_cuda = device.startswith("cuda")
	else:
		on_cuda = device.type == "cuda"

	if not on_cuda:
		return None

	key = amp_dtype.lower()
	if key == "auto":
		return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
	if key in ("bf16", "bfloat16"):
		return torch.bfloat16
	if key in ("fp16", "float16"):
		return torch.float16
	if key in ("off", "none"):
		return None
	raise ValueError(
		f"Unknown amp_dtype {amp_dtype!r}. Use auto, bf16, fp16, or off."
	)


def autocast_context(device, amp_dtype: str):
	"""Return a torch.amp.autocast context (or nullcontext on CPU / off)."""
	dtype = resolve_amp_dtype(amp_dtype, device)
	if dtype is None:
		return contextlib.nullcontext()
	return torch.amp.autocast("cuda", dtype=dtype)


def amp_dtype_label(amp_dtype: str, device) -> str:
	"""Human-readable label for logging which AMP mode is active."""
	resolved = resolve_amp_dtype(amp_dtype, device)
	if resolved is None:
		return "off"
	return "bf16" if resolved is torch.bfloat16 else "fp16"


import math

import torch
from torch import nn


class GoalActor(nn.Module):
    def __init__(self, d_model=192, action_dim=7, block_size=5,
                 nlayers=3, nheads=4, ff_dim=768, dropout=0.1):
        super().__init__()
        self.d_model, self.action_dim, self.block_size = d_model, action_dim, block_size
        self.config = dict(d_model=d_model, action_dim=action_dim, block_size=block_size,
                           nlayers=nlayers, nheads=nheads, ff_dim=ff_dim, dropout=dropout)
        self.context = nn.ModuleList([nn.Linear(d_model, d_model) for _ in range(3)])
        self.previous = nn.Sequential(nn.Linear(block_size * action_dim, 64), nn.GELU(),
                                      nn.Linear(64, d_model))
        self.action = nn.Linear(action_dim, d_model)
        self.bos = nn.Parameter(torch.zeros(1, 1, d_model))
        self.position = nn.Parameter(torch.randn(1, 4 + block_size, d_model) * 0.02)
        layer = nn.TransformerEncoderLayer(d_model, nheads, ff_dim, dropout,
                                           activation='gelu', batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(layer, nlayers, norm=nn.LayerNorm(d_model),
                                                 enable_nested_tensor=False)
        self.mean = nn.Linear(d_model, action_dim)
        self.log_std = nn.Linear(d_model, action_dim)

    def forward(self, current, goal, previous, prefix):
        """prefix [B,L,7] predicts L+1 actions (max five), using only past tokens."""
        if prefix.shape[1] >= self.block_size:
            raise ValueError('Prefix must be shorter than the action block')
        intent = goal - current
        context = torch.stack([p(x) for p, x in zip(
            self.context, (current, intent, current * intent))] +
            [self.previous(previous.flatten(1))], dim=1)
        tokens = torch.cat((context, self.bos.expand(current.shape[0], -1, -1),
                            self.action(prefix)), dim=1)
        tokens = tokens + self.position[:, :tokens.shape[1]]
        causal = torch.ones(tokens.shape[1], tokens.shape[1], dtype=torch.bool,
                            device=tokens.device).triu(1)
        hidden = self.transformer(tokens, mask=causal)[:, 4:]
        return self.mean(hidden), self.log_std(hidden).clamp(-5, 2)

    @torch.no_grad()
    def generate(self, current, goal, previous):
        prefix = current.new_empty(current.shape[0], 0, self.action_dim)
        for _ in range(self.block_size):
            mean, _ = self(current, goal, previous, prefix)
            prefix = torch.cat((prefix, mean[:, -1:]), dim=1)
        return prefix


def actor_objective(actor, current, goal, previous, targets, valid, student_probability=0.0):
    """Masked per-sample Gaussian NLL; Student Forcing only changes prefixes.

    Goal branch detached; current branch remains attached. No privileged state.
    Validation should pass student_probability=0 for deterministic teacher forcing.
    """
    prefix = targets[:, :-1]
    if student_probability > 0:
        was_training = actor.training
        actor.eval()
        try:
            generated = actor.generate(current.detach(), goal.detach(), previous.detach())
        finally:
            actor.train(was_training)
        select = torch.rand(targets.shape[0], 1, 1, device=targets.device) < student_probability
        prefix = torch.where(select, generated[:, :-1], prefix)
    mean, log_std = actor(current, goal.detach(), previous, prefix)
    mean, log_std, targets = mean.float(), log_std.float(), targets.float()
    nll = 0.5 * ((targets - mean) * torch.exp(-log_std)).square() + log_std + 0.5 * math.log(2 * math.pi)
    mask = valid.float()
    denominator = mask.sum(1).clamp_min(1)
    loss = ((nll.mean(-1) * mask).sum(1) / denominator).mean()
    mse = (((targets - mean).square().mean(-1) * mask).sum(1) / denominator).mean()
    return loss, mse


In [ ]:
import json
from pathlib import Path


def start_actor_wandb(options, *, training_resume, saved_run, output_dir, config):
    if not options or not options.get('enabled', True):
        return None
    import wandb
    mode = options.get('mode', 'auto')
    if mode not in ('auto', 'new', 'resume'):
        raise ValueError('WANDB_MODE must be auto, new, or resume')
    explicit_id = options.get('run_id') or None
    project = options.get('project', 'primary_actor')
    entity = options.get('entity') or None
    previous = saved_run
    sidecar = Path(output_dir) / 'wandb_run.json'
    if previous is None and training_resume and sidecar.exists():
        previous = json.loads(sidecar.read_text())
    should_resume = mode == 'resume' or (mode == 'auto' and training_resume and (explicit_id or previous))
    if should_resume and not training_resume:
        raise ValueError('Do not append scratch training to an old W&B run; use WANDB_MODE="new"')
    run_id = explicit_id
    if should_resume and not run_id:
        if not previous:
            raise ValueError('Set WANDB_RUN_ID to resume an existing W&B run')
        if previous['project'] != project or (entity and previous['entity'] != entity):
            raise ValueError('Saved W&B project/entity differs; choose WANDB_MODE="new" or an explicit run ID')
        run_id = previous['id']
        entity = previous['entity']
    if wandb.run is not None:
        raise RuntimeError('An active W&B run already exists. Finish it before starting training again.')
    # Read the Colab secret only at runtime. Never put the key in config/checkpoints.
    try:
        from google.colab import userdata
    except ImportError:
        # Local CLI: use `wandb login` or WANDB_API_KEY; never persist the key here.
        wandb.login()
    else:
        try:
            token = userdata.get('WANDB_TOKEN')
        except Exception as exc:
            raise RuntimeError('Enable access to WANDB_TOKEN in Colab, or disable W&B') from exc
        if not token:
            raise ValueError('WANDB_TOKEN is empty')
        wandb.login(key=token, relogin=True)
        del token
    run = wandb.init(project=project, entity=entity, name=options.get('name', 'primary_actor'),
                     id=run_id, resume='must' if should_resume else 'never',
                     config=config, mode='online', save_code=False, sync_tensorboard=False)
    try:
        run.define_metric('epoch')
        run.define_metric('*', step_metric='epoch')
        identity = dict(id=run.id, project=run.project, entity=run.entity, name=run.name)
        sidecar.parent.mkdir(parents=True, exist_ok=True)
        pending = sidecar.with_suffix('.json.tmp')
        pending.write_text(json.dumps(identity, indent=2))
        pending.replace(sidecar)
        print(f'W&B run: {run.url} (id={run.id})')
    except BaseException:
        run.finish(exit_code=1)
        raise
    return run


def log_actor_epoch(run, epoch, results, best_epoch, best_val_loss, learning_rate):
    if run is None:
        return
    metrics = {'epoch': epoch, 'optimization/learning_rate': learning_rate}
    for split in ('train', 'val'):
        for suffix in ('loss', 'prediction_loss', 'sigreg_loss', 'actor_nll', 'actor_mse'):
            metrics[f'{split}/{suffix}'] = results[f'{split}_{suffix}'][-1]
    for key in ('train_grad_norm', 'encoder_bn_gamma_rms', 'encoder_bn_running_var_min',
                'encoder_bn_running_var_max'):
        metrics[key] = results[key][-1]
    # Custom epoch axis, not explicit W&B step: continuing an existing run never
    # tries to write to a past internal step after a retry or logging fork.
    run.log(metrics)
    run.summary['best_epoch'] = best_epoch
    run.summary['best_validation_total'] = best_val_loss


import inspect
import math
import re
import shutil
import tempfile
import zipfile
from pathlib import Path

import torch
import yaml
from torch import nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm

CHECKPOINT_FORMAT_VERSION = 5
ARCHITECTURE_ID = "lewm_robot_goal_actor_action64_v5"
ACTION_ENCODING_ID = "standardized_ordered_primitive_block_v2"
IMAGE_NORMALIZATION_ID = "imagenet"



def load_config(config_path: Path) -> dict:
    with open(config_path) as f:
        return yaml.safe_load(f)


def _latest_epoch_checkpoint(save_dir: Path):
    checkpoints = []
    for path in save_dir.glob("epoch_*_model.zip"):
        match = re.fullmatch(r"epoch_(\d+)_model\.zip", path.name)
        if match:
            checkpoints.append((int(match.group(1)), path))
    return max(checkpoints, default=(None, None), key=lambda item: item[0])


def _load_training_state(
    path: Path, encoder, predictor, actor, optimizer, scaler, device, expected_metadata
):
    with zipfile.ZipFile(path, "r") as zf, tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        if "training_state.pth" not in zf.namelist():
            raise ValueError(f"{path} has no resumable training state")
        zf.extract("training_state.pth", tmp)
        try:
            training_state = torch.load(
                tmp / "training_state.pth", map_location=device, weights_only=False
            )
        except TypeError:
            training_state = torch.load(tmp / "training_state.pth", map_location=device)

        for key, expected in expected_metadata.items():
            actual = training_state.get(key)
            if actual != expected:
                raise ValueError(
                    f"Checkpoint {key}={actual!r}, expected {expected!r}. "
                    "Start a new run with the corrected architecture and action contract."
                )

        zf.extract("encoder_weights.pth", tmp)
        zf.extract("predictor_weights.pth", tmp)
        encoder.load_state_dict(
            torch.load(tmp / "encoder_weights.pth", map_location=device, weights_only=True)
        )
        predictor.load_state_dict(
            torch.load(tmp / "predictor_weights.pth", map_location=device, weights_only=True)
        )

        zf.extract("actor_weights.pth", tmp)
        actor.load_state_dict(torch.load(tmp / "actor_weights.pth", map_location=device, weights_only=True))
        try:
            optimizer.load_state_dict(training_state["optimizer"])
        except ValueError as exc:
            raise ValueError(
                "The checkpoint optimizer groups do not match the current optimizer. "
                "Start this run from scratch."
            ) from exc
        if training_state.get("scaler"):
            scaler.load_state_dict(training_state["scaler"])

    return training_state


def _save_checkpoint(
    zip_path: Path,
    encoder,
    predictor,
    actor,
    optimizer,
    scaler,
    epoch: int,
    state_mean: torch.Tensor,
    state_std: torch.Tensor,
    action_mean: torch.Tensor,
    action_std: torch.Tensor,
    action_low: torch.Tensor,
    action_high: torch.Tensor,
    checkpoint_metadata: dict,
    results: dict,
    config: dict | None,
    best_val_loss: float,
    best_epoch: int,
):
    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        encoder_path = tmp / "encoder_weights.pth"
        predictor_path = tmp / "predictor_weights.pth"
        training_path = tmp / "training_state.pth"
        state_stats_path = tmp / "state_stats.pth"

        torch.save(encoder.state_dict(), encoder_path)
        torch.save(predictor.state_dict(), predictor_path)
        torch.save(actor.state_dict(), tmp / "actor_weights.pth")
        torch.save(
            {
                "epoch": epoch,
                "optimizer": optimizer.state_dict(),
                "scaler": scaler.state_dict(),
                "state_mean": state_mean.cpu(),
                "state_std": state_std.cpu(),
                "action_mean": action_mean.cpu(),
                "action_std": action_std.cpu(),
                "action_low": action_low.cpu(),
                "action_high": action_high.cpu(),
                **checkpoint_metadata,
                "results": results,
                "best_val_loss": best_val_loss,
                "best_epoch": best_epoch,
            },
            training_path,
        )
        torch.save(
            {
                "state_mean": state_mean.cpu(),
                "state_std": state_std.cpu(),
                "action_mean": action_mean.cpu(),
                "action_std": action_std.cpu(),
                "action_low": action_low.cpu(),
                "action_high": action_high.cpu(),
                **checkpoint_metadata,
            },
            state_stats_path,
        )

        with zipfile.ZipFile(tmp / "checkpoint.zip", "w") as zf:
            zf.write(tmp / "actor_weights.pth", "actor_weights.pth")
            zf.write(encoder_path, "encoder_weights.pth")
            zf.write(predictor_path, "predictor_weights.pth")
            zf.write(training_path, "training_state.pth")
            zf.write(state_stats_path, "state_stats.pth")
            if config is not None:
                zf.writestr("config.yaml", yaml.safe_dump(config, sort_keys=False))
        _atomic_copy(tmp / "checkpoint.zip", zip_path)


def _atomic_copy(source: Path, destination: Path):
    pending = destination.with_suffix(destination.suffix + ".tmp")
    shutil.copyfile(source, pending)
    pending.replace(destination)


def _make_grad_scaler(enabled: bool):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


def _make_optimizer(
    named_parameters,
    lr,
    betas,
    weight_decay,
    on_cuda,
    no_decay_parameters,
):
    no_decay_parameter_ids = {id(parameter) for parameter in no_decay_parameters}
    decay_parameters = []
    no_decay_parameters = []
    for name, parameter in named_parameters:
        if not parameter.requires_grad:
            continue
        # Only BatchNorm affine parameters avoid decay. LayerNorm scales and
        # ordinary biases need the original decay to prevent the ViT runaway
        # observed when every one-dimensional parameter was exempted.
        if id(parameter) in no_decay_parameter_ids:
            no_decay_parameters.append(parameter)
        else:
            decay_parameters.append(parameter)

    kwargs = {
        "lr": lr,
        "betas": betas,
        "weight_decay": 0.0,
    }
    if on_cuda and "fused" in inspect.signature(torch.optim.AdamW).parameters:
        kwargs["fused"] = True
    optimizer = torch.optim.AdamW(
        [
            {"params": decay_parameters, "weight_decay": weight_decay},
            {"params": no_decay_parameters, "weight_decay": 0.0},
        ],
        **kwargs,
    )
    print(
        f"AdamW parameter groups: {len(decay_parameters)} decayed tensors, "
        f"{len(no_decay_parameters)} BatchNorm affine tensors without decay"
    )
    return optimizer


def _batch_norm_modules(*models):
    return [
        module
        for model in models
        for module in model.modules()
        if isinstance(module, nn.modules.batchnorm._BatchNorm)
    ]


@torch.no_grad()
def _recalibrate_batch_norm(
    encoder,
    predictor,
    dataloader,
    max_batches,
    device,
    non_blocking,
    amp_dtype,
):
    batch_norms = _batch_norm_modules(encoder, predictor)
    if max_batches <= 0 or not batch_norms:
        encoder.eval()
        predictor.eval()
        return 0

    encoder.eval()
    predictor.eval()
    momenta = [module.momentum for module in batch_norms]
    for module in batch_norms:
        module.reset_running_stats()
        # Use a cumulative average during calibration so the result does not
        # depend disproportionately on the final few shuffled batches.
        module.momentum = None
        module.train()

    batches_seen = 0
    try:
        for batch in dataloader:
            if batches_seen >= max_batches:
                break
            above, wrist, state, actions = _prepare_batch(
                batch, device, non_blocking
            )
            with autocast_context(device, amp_dtype):
                embeddings = encoder(above, wrist, state)
                predictor(embeddings[:, :3], actions)
            batches_seen += 1
    finally:
        for module, momentum in zip(batch_norms, momenta):
            module.momentum = momentum
        encoder.eval()
        predictor.eval()

    return batches_seen


def _batch_norm_diagnostics(model):
    batch_norms = _batch_norm_modules(model)
    if not batch_norms:
        return float("nan"), float("nan"), float("nan")

    scales = torch.cat(
        [module.weight.detach().float().flatten() for module in batch_norms]
    )
    running_vars = torch.cat(
        [module.running_var.detach().float().flatten() for module in batch_norms]
    )
    return (
        scales.square().mean().sqrt().item(),
        running_vars.min().item(),
        running_vars.max().item(),
    )


def _loader_options(num_workers, persistent_workers, pin_memory,
                    prefetch_factor, in_order, timeout, start_method):
    options = {
        "num_workers": num_workers,
        "persistent_workers": persistent_workers and num_workers > 0,
        "pin_memory": pin_memory,
    }
    if num_workers > 0:
        options["prefetch_factor"] = prefetch_factor
        options["timeout"] = timeout
        options["multiprocessing_context"] = start_method
    if "in_order" in inspect.signature(DataLoader).parameters:
        options["in_order"] = in_order
    return options


def _prepare_batch(batch, device, non_blocking):
    above, wrist, state, actions = batch[:4]
    # Keep images as uint8 in workers and pinned host memory. Conversion on the
    # GPU cuts host memory and transfer traffic by 4x.
    above = above.to(device, non_blocking=non_blocking).float().mul_(1.0 / 255.0)
    wrist = wrist.to(device, non_blocking=non_blocking).float().mul_(1.0 / 255.0)
    above = normalize_robot_images(above)
    wrist = normalize_robot_images(wrist)
    state = state.to(device, non_blocking=non_blocking)
    actions = actions.to(device, non_blocking=non_blocking)
    if actions.ndim != 4:
        raise ValueError(f"expected ordered action blocks [B, W, S, A], got {actions.shape}")
    actions = actions.flatten(start_dim=2)
    return above, wrist, state, actions


def train(device,
          data_dir: Path,
          log_dir: Path,
          save_dir: Path,
          frame_skip: int,
          lr: float,
          seed: int,
          lambd: float,
          epochs: int,
          batch_size: int,
          num_workers: int,
          persistent_workers: bool,
          pin_memory: bool,
          prefetch_factor: int,
          in_order: bool,
          loader_timeout: float,
          loader_start_method: str,
          compile_model: bool,
          window_size: int,
          embedding_dim: int,
          action_dim: int,
          pred_nheads: int,
          dropout: float,
          pred_nlayers: int,
          pred_dim_head: int,
          pred_mlp_dim: int,
          action_smoothing_dim: int,
          projector_hidden_dim: int,
          patch_size: int,
          enc_nheads: int,
          enc_nlayers: int,
          enc_npatches: int,
          knots: int,
          num_proj: int,
          optimizer_betas: tuple[float, float],
          optimizer_weight_decay: float,
          gradient_clip_norm: float,
          bn_calibration_batches: int,
          amp_dtype: str,
          resume: bool = False,
          config: dict | None = None,
          actor_weight: float = 0.1,
          student_probability: float = 0.5,
          history_dropout: float = 0.1,
          wandb_options: dict | None = None):

    if not math.isfinite(actor_weight) or actor_weight <= 0:
        raise ValueError('actor_weight must be finite and positive')
    if not 0 <= student_probability <= 1 or not 0 <= history_dropout <= 1:
        raise ValueError('Student forcing and history dropout must be probabilities')
    if window_size != 3 or frame_skip != 5 or action_dim != 7:
        raise ValueError('Actor dataset requires window=3, frame_skip=5, action_dim=7')
    if action_smoothing_dim != 64:
        raise ValueError('This training notebook requires action_smoothing_dim=64')
    if resume:
        if _latest_epoch_checkpoint(save_dir)[1] is None:
            raise FileNotFoundError(f'No checkpoint to resume in {save_dir}')
    elif any(save_dir.glob('*.zip')):
        raise ValueError('Training checkpoints already exist; use RESUME=True or a new DRIVE_DIR')
    torch.manual_seed(seed)
    if isinstance(device, str):
        on_cuda = device.startswith("cuda")
    else:
        on_cuda = device.type == "cuda"

    if on_cuda:
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.benchmark = True
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    save_dir.mkdir(parents=True, exist_ok=True)

    # Calculate these once from the train mask, then use the same values for
    # both splits and save them with every checkpoint.
    h5_path = Path(data_dir) / "robot" / "actor" / "demo_actor.hdf5"
    state_mean, state_std = compute_train_state_stats(h5_path)
    action_mean, action_std, action_low, action_high = compute_train_action_stats(h5_path)
    print(f"state mean: {state_mean.tolist()}")
    print(f"state std:  {state_std.tolist()}")
    print(f"action mean: {action_mean.tolist()}")
    print(f"action std:  {action_std.tolist()}")
    print(
        f"temporal stride: frame_skip={frame_skip} "
        f"({frame_skip} simulator steps per model transition)"
    )

    train_dataset = RobotActorDataset(
        data_dir=data_dir,
        frame_skip=frame_skip,
        window=window_size,
        val=False,
        state_mean=state_mean,
        state_std=state_std,
        action_mean=action_mean,
        action_std=action_std,
    )
    val_dataset = RobotActorDataset(
        data_dir=data_dir,
        frame_skip=frame_skip,
        window=window_size,
        val=True,
        state_mean=state_mean,
        state_std=state_std,
        action_mean=action_mean,
        action_std=action_std,
    )

    predictor_action_dim = frame_skip * action_dim
    checkpoint_metadata = {
        "checkpoint_format_version": CHECKPOINT_FORMAT_VERSION,
        "architecture_id": ARCHITECTURE_ID,
        "action_encoding_id": ACTION_ENCODING_ID,
        "image_normalization_id": IMAGE_NORMALIZATION_ID,
        "frame_skip": frame_skip,
        "primitive_action_dim": action_dim,
        "predictor_action_dim": predictor_action_dim,
        "action_smoothing_dim": action_smoothing_dim,
        "initialization": "scratch",
        "training_objective": "prediction_sigreg_goal_actor_nll_v1",
        "actor_weight": float(actor_weight),
        "student_probability": float(student_probability),
        "history_dropout": float(history_dropout),
        "actor_config": GoalActor(d_model=embedding_dim).config,
        "dataset_revision": (config or {}).get("dataset_revision"),
    }

    loader_options = _loader_options(
        num_workers,
        persistent_workers,
        pin_memory,
        prefetch_factor,
        in_order,
        loader_timeout,
        loader_start_method,
    )
    print(
        f"DataLoader: workers={num_workers}, start_method="
        f"{loader_options.get('multiprocessing_context', 'single-process')}, "
        f"prefetch={loader_options.get('prefetch_factor', 0)}, "
        f"persistent={loader_options['persistent_workers']}"
    )
    train_dataloader = DataLoader(
        dataset=train_dataset,
        batch_size=batch_size,
        shuffle=True,
        **loader_options,
    )
    val_dataloader = DataLoader(
        dataset=val_dataset,
        batch_size=batch_size,
        shuffle=True,
        **loader_options,
    )

    predictor = PredictorNetwork(
        d_model=embedding_dim,
        nheads=pred_nheads,
        d_action=predictor_action_dim,
        dropout=dropout,
        nlayers=pred_nlayers,
        window=window_size,
        dim_head=pred_dim_head,
        mlp_dim=pred_mlp_dim,
        action_smoothing_dim=action_smoothing_dim,
        projector_hidden_dim=projector_hidden_dim,
    ).to(device)
    encoder = RobotEncoder(
        patch=patch_size,
        nheads=enc_nheads,
        nlayers=enc_nlayers,
        d_model=embedding_dim,
        npatches=enc_npatches,
        projector_hidden_dim=projector_hidden_dim,
    ).to(device)

    actor = GoalActor(d_model=embedding_dim).to(device)
    gpu_name = torch.cuda.get_device_name(0) if on_cuda else "cpu"
    amp_label = amp_dtype_label(amp_dtype, device)
    print(f"networks are on device: {device} ({gpu_name})")
    print(f"AMP: {amp_label} (config amp_dtype={amp_dtype})")

    sig_reg = SIGReg(knots=knots, num_proj=num_proj).to(device)
    pred_loss_fn = nn.MSELoss()
    named_parameters = [
        (f"predictor.{name}", parameter)
        for name, parameter in predictor.named_parameters()
    ] + [
        (f"encoder.{name}", parameter)
        for name, parameter in encoder.named_parameters()
    ]
    named_parameters += [(f"actor.{name}", p) for name, p in actor.named_parameters()]
    trainable_parameters = [
        parameter for _, parameter in named_parameters if parameter.requires_grad
    ]
    batch_norm_parameters = [
        parameter
        for module in _batch_norm_modules(predictor, encoder)
        for parameter in module.parameters(recurse=False)
        if parameter.requires_grad
    ]
    optimizer = _make_optimizer(
        named_parameters,
        lr,
        optimizer_betas,
        optimizer_weight_decay,
        on_cuda,
        batch_norm_parameters,
    )
    scaler = _make_grad_scaler(enabled=on_cuda and amp_label == "fp16")

    writer = SummaryWriter(log_dir=log_dir)
    wb_run = None
    try:
        results = {
            "train_loss": [],
            "train_prediction_loss": [],
            "train_sigreg_loss": [],
            "train_actor_nll": [],
            "train_actor_mse": [],
            "train_grad_norm": [],
            "val_loss": [],
            "val_prediction_loss": [],
            "val_sigreg_loss": [],
            "val_actor_nll": [],
            "val_actor_mse": [],
            "encoder_bn_gamma_rms": [],
            "encoder_bn_running_var_min": [],
            "encoder_bn_running_var_max": [],
        }
        saved_wandb = None
        start_epoch = 0
        best_val_loss = float("inf")
        best_epoch = -1

        if resume:
            last_epoch, checkpoint_path = _latest_epoch_checkpoint(save_dir)
            if checkpoint_path is None:
                print(f"No epoch_*_model.zip found in {save_dir}; starting from scratch")
            else:
                training_state = _load_training_state(
                    checkpoint_path,
                    encoder,
                    predictor,
                    actor,
                    optimizer,
                    scaler,
                    device,
                    checkpoint_metadata,
                )
                if not training_state:
                    raise ValueError(
                        f"{checkpoint_path} contains weights only and cannot resume "
                        "optimizer state. Start a new run or use a new-format checkpoint."
                    )
                saved_mean = training_state["state_mean"].cpu()
                saved_std = training_state["state_std"].cpu()
                if not torch.allclose(saved_mean, state_mean) or not torch.allclose(saved_std, state_std):
                    raise ValueError("Dataset state statistics do not match the checkpoint")
                for key, current in (
                    ("action_mean", action_mean),
                    ("action_std", action_std),
                    ("action_low", action_low),
                    ("action_high", action_high),
                ):
                    saved = training_state.get(key)
                    if saved is None or not torch.allclose(saved.cpu(), current):
                        raise ValueError(f"Dataset {key} does not match the checkpoint")
                saved_wandb = training_state.get("wandb_run")
                results = training_state.get("results", results)
                start_epoch = int(training_state.get("epoch", last_epoch)) + 1
                previous_val_losses = results.get("val_loss", [])
                best_val_loss = float(
                    training_state.get(
                        "best_val_loss",
                        min(previous_val_losses, default=float("inf")),
                    )
                )
                best_epoch = int(training_state.get("best_epoch", -1))
                if best_epoch < 0 and previous_val_losses:
                    best_epoch = min(
                        range(len(previous_val_losses)),
                        key=previous_val_losses.__getitem__,
                    )
                print(f"resumed {checkpoint_path.name}; continuing at epoch {start_epoch}")

        wb_run = start_actor_wandb(
            wandb_options, training_resume=resume, saved_run=saved_wandb,
            output_dir=save_dir.parent,
            config={**(config or {}), **checkpoint_metadata, "learning_rate": lr,
                    "batch_size": batch_size, "sigreg_weight": lambd,
                    "additional_epochs": epochs, "start_epoch": start_epoch,
                    "num_workers": num_workers, "amp_dtype": amp_dtype},
        )
        if wb_run is not None:
            checkpoint_metadata["wandb_run"] = dict(
                id=wb_run.id, project=wb_run.project, entity=wb_run.entity, name=wb_run.name)
        elif saved_wandb:
            checkpoint_metadata["wandb_run"] = saved_wandb

        if compile_model and on_cuda:
            if hasattr(encoder, "compile") and hasattr(predictor, "compile"):
                encoder.compile()
                predictor.compile()
                print("torch.compile enabled for encoder and predictor")
            else:
                print("torch.compile requested but unavailable; using eager mode")

        non_blocking = on_cuda and pin_memory

        print(f"Training epochs [{start_epoch}, {start_epoch + epochs})")
        for epoch in tqdm(range(start_epoch, start_epoch + epochs)):
            print(f"Epoch: {epoch}\n------------")

            train_metrics = torch.zeros(6, device=device)

            encoder.train()
            predictor.train()
            actor.train()
            for batch in tqdm(train_dataloader, desc="Train batches", leave=False):
                above, wrist, state, actions = _prepare_batch(
                    batch, device, non_blocking
                )

                optimizer.zero_grad(set_to_none=True)
                with autocast_context(device, amp_dtype):
                    # Encode o_t ... o_{t+window} once, then use the same shifted
                    # prediction objective as the original LeWM training process.
                    all_embeddings = encoder(above, wrist, state)
                    pred_embeddings = predictor(all_embeddings[:, :3], actions)
                    target_embeddings = all_embeddings[:, 1:4]

                    pred_loss = pred_loss_fn(pred_embeddings, target_embeddings)
                    reg_loss = sig_reg(all_embeddings[:, :4].transpose(0, 1))
                    valid = batch[4].to(device, non_blocking=non_blocking)
                    previous = actions[:, 1].reshape(-1, frame_skip, action_dim)
                    # Match rollout initialization: no demonstration actions at reset.
                    zero_history = (-action_mean.to(device) / action_std.to(device)).expand_as(previous)
                    missing = torch.rand(previous.shape[0], 1, 1, device=device) < history_dropout
                    previous = torch.where(missing, zero_history, previous)
                    actor_loss, actor_mse = actor_objective(
                        actor, all_embeddings[:, 2], all_embeddings[:, 4], previous,
                        actions[:, 2].reshape(-1, frame_skip, action_dim), valid, student_probability
                    )
                    loss = pred_loss + lambd * reg_loss + actor_weight * actor_loss

                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                grad_norm = torch.nn.utils.clip_grad_norm_(
                    trainable_parameters,
                    max_norm=gradient_clip_norm,
                    error_if_nonfinite=True,
                )
                scaler.step(optimizer)
                scaler.update()

                train_metrics += torch.stack(
                    (
                        loss.detach(),
                        pred_loss.detach(),
                        reg_loss.detach(),
                        actor_loss.detach(),
                        actor_mse.detach(),
                        grad_norm.detach(),
                    )
                )

            num_train_batches = len(train_dataloader)
            train_total, train_prediction, train_sigreg, train_actor, train_valid, train_grad_norm = (
                train_metrics.div(num_train_batches).cpu().tolist()
            )

            calibration_batches_seen = _recalibrate_batch_norm(
                encoder=encoder,
                predictor=predictor,
                dataloader=train_dataloader,
                max_batches=bn_calibration_batches,
                device=device,
                non_blocking=non_blocking,
                amp_dtype=amp_dtype,
            )
            bn_gamma_rms, bn_var_min, bn_var_max = _batch_norm_diagnostics(encoder)

            val_metrics = torch.zeros(5, device=device)

            encoder.eval()
            predictor.eval()
            actor.eval()
            with torch.inference_mode():
                for batch in val_dataloader:
                    above, wrist, state, actions = _prepare_batch(
                        batch, device, non_blocking
                    )

                    with autocast_context(device, amp_dtype):
                        all_embeddings = encoder(above, wrist, state)
                        pred_embeddings = predictor(all_embeddings[:, :3], actions)
                        target_embeddings = all_embeddings[:, 1:4]

                        pred_loss = pred_loss_fn(pred_embeddings, target_embeddings)
                        reg_loss = sig_reg(all_embeddings[:, :4].transpose(0, 1))
                        valid = batch[4].to(device, non_blocking=non_blocking)
                        actor_loss, actor_mse = actor_objective(
                            actor, all_embeddings[:, 2], all_embeddings[:, 4],
                            actions[:, 1].reshape(-1, frame_skip, action_dim),
                            actions[:, 2].reshape(-1, frame_skip, action_dim), valid, 0.0
                        )
                        loss = pred_loss + lambd * reg_loss + actor_weight * actor_loss

                    val_metrics += torch.stack((loss, pred_loss, reg_loss, actor_loss, actor_mse))

            if not torch.isfinite(val_metrics).all():
                raise FloatingPointError("Nonfinite validation losses; checkpoint not saved")
            num_val_batches = len(val_dataloader)
            val_total, val_prediction, val_sigreg, val_actor, val_valid = (
                val_metrics.div(num_val_batches).cpu().tolist()
            )

            results["train_loss"].append(train_total)
            results["train_prediction_loss"].append(train_prediction)
            results["train_sigreg_loss"].append(train_sigreg)
            results["train_actor_nll"].append(train_actor)
            results["train_actor_mse"].append(train_valid)
            results.setdefault("train_grad_norm", []).append(train_grad_norm)
            results["val_loss"].append(val_total)
            results["val_prediction_loss"].append(val_prediction)
            results["val_sigreg_loss"].append(val_sigreg)
            results["val_actor_nll"].append(val_actor)
            results["val_actor_mse"].append(val_valid)
            results.setdefault("encoder_bn_gamma_rms", []).append(bn_gamma_rms)
            results.setdefault("encoder_bn_running_var_min", []).append(bn_var_min)
            results.setdefault("encoder_bn_running_var_max", []).append(bn_var_max)

            writer.add_scalars(
                main_tag="Loss/Total",
                tag_scalar_dict={"train": train_total, "validation": val_total},
                global_step=epoch,
            )
            writer.add_scalars(
                main_tag="Loss/Prediction",
                tag_scalar_dict={"train": train_prediction, "validation": val_prediction},
                global_step=epoch,
            )
            writer.add_scalars(
                main_tag="Loss/SIGReg",
                tag_scalar_dict={"train": train_sigreg, "validation": val_sigreg},
                global_step=epoch,
            )
            writer.add_scalars(
                main_tag="Loss/ActorNLL",
                tag_scalar_dict={"train": train_actor, "validation": val_actor},
                global_step=epoch,
            )
            writer.add_scalars(
                main_tag="Loss/ActorMSE",
                tag_scalar_dict={"train": train_valid, "validation": val_valid},
                global_step=epoch,
            )
            writer.add_scalar("Optimization/GradientNorm", train_grad_norm, epoch)
            writer.add_scalar("BatchNorm/EncoderGammaRMS", bn_gamma_rms, epoch)
            writer.add_scalar("BatchNorm/EncoderRunningVarMin", bn_var_min, epoch)
            writer.add_scalar("BatchNorm/EncoderRunningVarMax", bn_var_max, epoch)

            is_best = val_total < best_val_loss
            if is_best:
                best_val_loss = val_total
                best_epoch = epoch

            zip_path = save_dir / f"epoch_{epoch}_model.zip"
            _save_checkpoint(
                zip_path=zip_path,
                encoder=encoder,
                predictor=predictor,
                actor=actor,
                optimizer=optimizer,
                scaler=scaler,
                epoch=epoch,
                state_mean=state_mean,
                state_std=state_std,
                action_mean=action_mean,
                action_std=action_std,
                action_low=action_low,
                action_high=action_high,
                checkpoint_metadata=checkpoint_metadata,
                results=results,
                config=config,
                best_val_loss=best_val_loss,
                best_epoch=best_epoch,
            )
            if is_best:
                _atomic_copy(zip_path, save_dir / "best_model.zip")
            writer.flush()
            log_actor_epoch(wb_run, epoch, results, best_epoch, best_val_loss, optimizer.param_groups[0]["lr"])

            print(
                f"Train total: {train_total:.4f} "
                f"(prediction {train_prediction:.4f}, SIGReg {train_sigreg:.4f}, "
                f"actor NLL {train_actor:.4f}) | "
                f"Validation total: {val_total:.4f} "
                f"(prediction {val_prediction:.4f}, SIGReg {val_sigreg:.4f}, "
                f"actor NLL {val_actor:.4f}) | "
                f"grad norm {train_grad_norm:.4f} | "
                f"encoder BN gamma RMS {bn_gamma_rms:.4f} "
                f"(recalibrated on {calibration_batches_seen} batches)"
            )
            if is_best:
                print(f"new best validation checkpoint → {save_dir / 'best_model.zip'}")

        return results



    finally:
        train_dataset.close()
        val_dataset.close()
        writer.close()
        if wb_run is not None:
            import sys
            wb_run.finish(exit_code=1 if sys.exc_info()[0] else 0)


In [ ]:
import time

RESUME = False  # Restore the latest checkpoint and optimizer from DRIVE_DIR.
ENABLE_WANDB = True  # False keeps TensorBoard only.
WANDB_PROJECT = 'primary_actor'  # W&B project grouping.
WANDB_ENTITY = None  # Optional W&B user or team.
WANDB_RUN_NAME = 'primary_actor'  # Display name, not the unique run ID.
WANDB_MODE = 'auto'  # auto: follow RESUME; new: new history; resume: existing history.
WANDB_RUN_ID = None  # Optional explicit run ID; otherwise restored from checkpoint.

# All train() hyperparameters. Architecture/data-contract changes may require new data
# or matching inference settings. Resume restores checkpoint settings/optimizer state.
TRAINING = {
    'frame_skip': 5,  # Primitive steps per block; fixed at 5 by the dataset contract.
    'lr': 5e-05,  # AdamW learning rate.
    'seed': 42,  # Random seed.
    'lambd': 0.1,  # SIGReg loss weight.
    'epochs': 65,  # Additional epochs, also when resuming.
    'batch_size': 256,  # Samples per optimizer update.
    'num_workers': 2,  # Parallel data workers; tune to available CPU cores.
    'persistent_workers': False,  # Keep workers alive between epochs.
    'pin_memory': True,  # Pinned CPU memory for GPU transfers.
    'prefetch_factor': 2,  # Batches queued per worker.
    'in_order': True,  # Preserve loader batch ordering.
    'loader_timeout': 120,  # Seconds to wait for a worker batch; 0 disables timeout.
    'loader_start_method': 'spawn',  # Spawn avoids inherited HDF5/CUDA state.
    'compile_model': True,  # Compile encoder and predictor; first execution is slower.
    'window_size': 3,  # Observation history length; fixed at 3.
    'embedding_dim': 192,  # Latent feature width.
    'action_dim': 7,  # Primitive action width; fixed at 7.
    'pred_nheads': 16,  # Predictor attention heads.
    'dropout': 0.1,  # Predictor dropout probability.
    'pred_nlayers': 6,  # Predictor transformer layers.
    'pred_dim_head': 64,  # Width of each predictor attention head.
    'pred_mlp_dim': 2048,  # Predictor feedforward hidden width.
    'action_smoothing_dim': 64,  # Action-block projection width; this model requires 64.
    'projector_hidden_dim': 2048,  # Projection-head hidden width.
    'patch_size': 14,  # Vision patch side length; 14 for 84x84 images.
    'enc_nheads': 3,  # Attention heads in each camera encoder.
    'enc_nlayers': 12,  # Transformer layers in each camera encoder.
    'enc_npatches': 36,  # Image patches; (84 / patch_size)^2 = 36.
    'knots': 17,  # SIGReg integration points.
    'num_proj': 1024,  # SIGReg random projection count.
    'optimizer_betas': (0.9, 0.999),  # AdamW momentum coefficients.
    'optimizer_weight_decay': 0.001,  # AdamW weight decay.
    'gradient_clip_norm': 1.0,  # Maximum gradient norm.
    'bn_calibration_batches': 256,  # Batches for BatchNorm recalibration before validation.
    'amp_dtype': 'auto',  # auto, bf16, fp16, or off.
    'actor_weight': 0.1,  # Actor negative-log-likelihood loss weight.
    'student_probability': 0.5,  # Probability of using actor-generated training prefixes.
    'history_dropout': 0.1,  # Probability of dropping the previous action block.
}
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before training')
for key in ('epochs', 'batch_size'):
    if isinstance(TRAINING[key], bool) or not isinstance(TRAINING[key], int) or TRAINING[key] < 1:
        raise ValueError(f'{key} must be a positive integer')
if not DATASET_PATH.is_file():
    raise FileNotFoundError('Run the Hugging Face download cell first')
save_dir = DRIVE_DIR / 'models'
log_dir = DRIVE_DIR / 'logs'
if not RESUME and any(path.exists() and any(path.iterdir()) for path in (save_dir, log_dir)):
    raise ValueError(f'{DRIVE_DIR} already contains a run. Set RESUME=True or choose a new DRIVE_DIR')
print('Resuming latest checkpoint' if RESUME else 'Training from random initialization')
start_time = time.perf_counter()
results = train(
    device='cuda',
    data_dir=DATASET_PATH.parents[2],
    log_dir=log_dir,
    save_dir=save_dir,
    resume=RESUME,
    **TRAINING,
    wandb_options={'enabled': ENABLE_WANDB, 'project': WANDB_PROJECT,
                   'entity': WANDB_ENTITY, 'name': WANDB_RUN_NAME,
                   'mode': WANDB_MODE, 'run_id': WANDB_RUN_ID},
    config={**TRAINING, 'initialization': 'resume' if RESUME else 'scratch',
            'learning_rate': TRAINING['lr'], 'run_dir': str(DRIVE_DIR),
            'dataset_repo': HF_REPO, 'dataset_revision': DATASET_REVISION},
)
print(f'training time: {(time.perf_counter() - start_time):.2f}s')
print(f'checkpoints saved to: {save_dir}')
print(f'logs saved to: {log_dir}')


In [ ]:
%load_ext tensorboard
%tensorboard --logdir $log_dir
